# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Load tables
bookings = spark.table("bookings")
facilities = spark.table("facilities")
members = spark.table("members")

# Q1: How can you produce a list of the start times for bookings by members named 'David Farrell'?
print("Q1")
q1 = bookings.join(members, bookings.memid == members.memid).filter((members.firstname == "David") & (members.surname == "Farrell")).select(bookings.starttime)
q1.show()

# Q2: How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.
print("Q2")
q2 = bookings.join(facilities, bookings.facid == facilities.facid).filter((facilities.name.like("%Tennis Court%")) & (to_date(bookings.starttime) == "2012-09-21")).select(bookings.starttime, facilities.name).orderBy(bookings.starttime)
q2.show()

# Q3: How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).
print("Q3")
m1 = members.alias("m1")
m2 = members.alias("m2")
q3 = m1.join(m2, m1.recommendedby == m2.memid, "left").select(m1.surname, m1.firstname, m2.surname.alias("recommender_surname"), m2.firstname.alias("recommender_firstname")).orderBy(m1.surname, m1.firstname)
q3.show()

# Q4: How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.
print("Q4")
q4 = bookings.join(members, bookings.memid == members.memid).join(facilities, bookings.facid == facilities.facid).filter(facilities.name.like("%Tennis Court%")).select(concat(members.firstname, lit(" "), members.surname).alias("member_name"), facilities.name.alias("court_name")).distinct().orderBy("member_name", "court_name")
q4.show()

# Q5: How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.
print("Q5")
rec = members.select(col("memid").alias("rid"), concat(col("firstname"), lit(" "), col("surname")).alias("recommended_by"))
q5 = members.select(concat(col("firstname"), lit(" "), col("surname")).alias("member_name"), col("recommendedby")).join(rec, members.recommendedby == rec.rid, "left").select("member_name", "recommended_by").distinct().orderBy("member_name")
q5.show()

# Q6: Produce a count of the number of recommendations each member has made. Order by member ID.
print("Q6")
q6 = members.groupBy("recommendedby").count().filter(col("recommendedby").isNotNull()).orderBy("recommendedby")
q6.show()

# Q7: Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.
print("Q7")
q7 = bookings.groupBy("facid").agg(sum("slots").alias("total_slots")).orderBy("facid")
q7.show()

# Q8: Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.
print("Q8")
q8 = bookings.filter((month("starttime") == 9) & (year("starttime") == 2012)).groupBy("facid").agg(sum("slots").alias("total_slots")).orderBy("total_slots")
q8.show()

# Q9: Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.
print("Q9")
q9 = bookings.filter(year("starttime") == 2012).groupBy("facid", month("starttime").alias("month")).agg(sum("slots").alias("total_slots")).orderBy("facid", "month")
q9.show()

# Q10: Find the total number of members (including guests) who have made at least one booking.
print("Q10")
q10 = bookings.select("memid").distinct()
print(q10.count())

# Q11: Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.
print("Q11")
q11 = bookings.filter(col("starttime") > "2012-09-01").groupBy("memid").agg(min("starttime").alias("first_booking")).join(members, "memid").select("memid", concat(col("firstname"), lit(" "), col("surname")).alias("member_name"), "first_booking").orderBy("memid")
q11.show()

# Q12: Output the names of all members, formatted as 'Surname, Firstname'
print("Q12")
q12 = members.select(concat(col("surname"), lit(", "), col("firstname")).alias("member_name"))
q12.show()

# Q13: Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.
print("Q13")
q13 = facilities.filter(lower(col("name")).startswith("tennis"))
q13.show()

# Q14: You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.
print("Q14")
q14 = members.filter(col("telephone").contains("(")).select("memid", "telephone").orderBy("memid")
q14.show()

# Q15 : You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.
print("Q15")
q15 = members.groupBy(substring(col("surname"), 1, 1).alias("first_letter")).count().orderBy("first_letter")
q15.show()

# Q16: Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.
print("Q16")
q16 = spark.sql("SELECT explode(sequence(to_date('2012-10-01'), to_date('2012-10-31'), interval 1 day)) as date")
q16.show()

# Q17 : Return a count of bookings for each month, sorted by month
print("Q17")
q17 = bookings.groupBy(month(col("starttime")).alias("month")).count().orderBy("month")
q17.show()

Q1
+-------------------+
|          starttime|
+-------------------+
|2012-09-18 09:00:00|
|2012-09-18 17:30:00|
|2012-09-18 13:30:00|
|2012-09-18 20:00:00|
|2012-09-19 09:30:00|
|2012-09-19 15:00:00|
|2012-09-19 12:00:00|
|2012-09-20 15:30:00|
|2012-09-20 11:30:00|
|2012-09-20 14:00:00|
|2012-09-21 10:30:00|
|2012-09-21 14:00:00|
|2012-09-22 08:30:00|
|2012-09-22 17:00:00|
|2012-09-23 08:30:00|
|2012-09-23 17:30:00|
|2012-09-23 19:00:00|
|2012-09-24 08:00:00|
|2012-09-24 16:30:00|
|2012-09-24 12:30:00|
+-------------------+
only showing top 20 rows
Q2
+-------------------+--------------+
|          starttime|          name|
+-------------------+--------------+
|2012-09-21 08:00:00|Tennis Court 2|
|2012-09-21 08:00:00|Tennis Court 1|
|2012-09-21 09:30:00|Tennis Court 1|
|2012-09-21 10:00:00|Tennis Court 2|
|2012-09-21 11:30:00|Tennis Court 2|
|2012-09-21 12:00:00|Tennis Court 1|
|2012-09-21 13:30:00|Tennis Court 1|
|2012-09-21 14:00:00|Tennis Court 2|
|2012-09-21 15:30:00|Tennis Court 